## Import Library

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

## Load Data

In [ ]:
df = pd.read_excel("./dataset/sales2.xlsx", sheet_name="Sale1")
df.head()

## Data Preprocessing

In [ ]:
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"])

# check missing value
print("Missing value: ", df.isnull().sum().sum())

# fill Nan with 0 for numerical data
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(0)

# fill Nan with unknow for numerical data
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
df[cat_cols] = df[cat_cols].fillna("Unknown")

# Encode categorical with the One Hot Encoding
categorical_cols = ['DayType', 'Weather_8am_1pm', 'Weather_2pm_7pm', 'Weather_8pm_11pm', 'Traffic_Level']
encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown='ignore')

encoded_array = encoder.fit_transform(df[categorical_cols])

# Create DataFrame for encoded features
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(categorical_cols), index=df.index)
# print("columns: ", encoder.get_feature_names_out(categorical_cols))
# Drop original categorical columns & join encoded ones
df = df.drop(columns=categorical_cols)
df = pd.concat([df, encoded_df], axis=1)

print("Data shape after encoding:", df.shape)
df


## Feature Enginering

In [ ]:
# Time based features
if "Date" in df.columns:
    df['DayOfWeek'] = df['Date'].dt.dayofweek #0 = isnin and 6=ahad
    df['Month'] = df['Date'].dt.month
    df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
    df['Quater'] = df['Date'].dt.quarter
    df['IsWeekend'] = df['Date'].isin([5, 6]).astype(int)
    df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
    df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)
    
    
# lag feature
target_cols = [
    "Drink_IcedCaramelMacchiato", "Drink_MochaFrappe", "Drink_StrawberrySmoothie",
    "Drink_MangoPassionfruitCooler", "Drink_MatchaLatte", "Drink_ChocolateHazelnutShake",
    "Drink_BlueberryLemonadeSparkler", "Drink_AvocadoCoffee"
]

for col in target_cols:
    df[f'{col}_lag1'] = df[col].shift(1) #sales cup yesterdat
    df[f'{col}_lag7'] = df[col].shift(7) #sales cup last 7 days
    
# Rolling feature
for col in target_cols:
    df[f'{col}_rolling3_mean'] = df[col].shift(1).rolling(window=3).mean() #last 3 day avg
    df[f'{col}_rolling7_mean'] = df[col].shift(1).rolling(window=7).mean() #last 7 day avg
    df[f'{col}_rolling7_sum'] = df[col].shift(1).rolling(window=7).sum() #last 3 day sum
    
# Aggregate feature
promo_col = [c for c in df.columns if "Promo_" in c]
df["Total_Promo_Active"] = df[promo_col].sum(axis=1) #horizontal

# Total events flags(SchoolHoliday, PublicHoliday, Festival)
event_cols = ["SchoolHoliday", "PublicHoliday", "Festival", "BackToSchool", "ExamSeason", "PayDay", "BigEventNearby"]
event_cols = [col for col in event_cols if col in df.columns]
df["Total_Event_Flag"] = df[event_cols].sum(axis=1)

# Previous day total cups sold
if "Cup_Total" in df.columns:
    df["PrevDay_CupTotal"] = df["Cup_Total"].shift(1)
    df["Rolling7_CupTotal"] = df["Cup_Total"].shift(1).rolling(7).sum()

# Drop rows with NaN (caused by lag/rolling)
df = df.dropna().reset_index(drop=True)

print("Data shape after feature engineering:", df.shape)
df.head()

In [ ]:
df.columns

## Define Feature & Target, split

In [ ]:
# no need column
df.drop(columns=['Date', "Day", "Cup_Total"], inplace=True)

#target/y
target_cols = [
    "Drink_IcedCaramelMacchiato",
    "Drink_MochaFrappe",
    "Drink_StrawberrySmoothie",
    "Drink_MangoPassionfruitCooler",
    "Drink_MatchaLatte",
    "Drink_ChocolateHazelnutShake",
    "Drink_BlueberryLemonadeSparkler",
    "Drink_AvocadoCoffee"
]

X = df.drop(columns=target_cols)
y = df[target_cols]

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

In [ ]:
df.columns

## Model Training

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

# Hyperparameter grid
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.05, 0.1, 0.2, 0.3],
    "max_depth": [2, 3, 4],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 3, 5]
}

# Dictionary untuk simpan best model & results
best_models = {}
results_gb_grid = {}

# Loop untuk setiap target drink
for col in y_train.columns:
    print(f"\n🔎 GridSearch untuk: {col}")
    
    gb = GradientBoostingRegressor(random_state=42)
    
    grid_search = GridSearchCV(
        estimator=gb,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",  # minimize MAE
        cv=3,  # 3-fold cross-validation
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train[col])
    
    # Simpan best model
    best_models[col] = grid_search.best_estimator_
    print(f"✅ Best params untuk {col}: {grid_search.best_params_}")
    
    # Prediction
    y_pred = best_models[col].predict(X_test)
    
    # Evaluation metrics
    mae = mean_absolute_error(y_test[col], y_pred)
    mse = mean_squared_error(y_test[col], y_pred)
    r2 = r2_score(y_test[col], y_pred)
    
    results_gb_grid[col] = {
        "Best_Params": grid_search.best_params_,
        "MAE": mae,
        "MSE": mse,
        "R2": r2
    }

# Convert result ke DataFrame
results_gb_grid_df = pd.DataFrame(results_gb_grid).T
results_gb_grid_df.loc["Average", ["MAE", "MSE", "R2"]] = results_gb_grid_df[["MAE", "MSE", "R2"]].mean()
print("\n📊 Gradient Boosting (GridSearch) Results:")
print(results_gb_grid_df)


In [ ]:
# # Step 6: Model Training with Gradient Boosting

# from sklearn.ensemble import GradientBoostingRegressor
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# import numpy as np

# # Dictionary to simpan model dan result
# models_gb = {}
# results_gb = {}

# # Train Gradient Boosting untuk setiap drink (multi-output)
# for col in y_train.columns:
#     print(f"Training Gradient Boosting model for: {col}")
    
#     model = GradientBoostingRegressor(
#         n_estimators=200,   # bilangan pokok decision tree
#         learning_rate=0.5, # step size untuk update
#         max_depth=3,       # kedalaman pokok (balance under/overfitting)
#         random_state=42
#     )
    
#     model.fit(X_train, y_train[col])
#     models_gb[col] = model
    
#     # prediction
#     y_pred = model.predict(X_test)
    
#     # evaluation metrics
#     mae = mean_absolute_error(y_test[col], y_pred)
#     mse = mean_squared_error(y_test[col], y_pred)
#     r2 = r2_score(y_test[col], y_pred)
    
#     results_gb[col] = {
#         "MAE": mae,
#         "MSE": mse,
#         "R2": r2
#     }

# # Convert results ke DataFrame untuk view senang
# import pandas as pd
# results_gb_df = pd.DataFrame(results_gb).T
# results_gb_df.loc['Average'] = results_gb_df.mean()
# print("\nGradient Boosting Results:")
# print(results_gb_df)


## Model Evaluation

In [ ]:
# from xgboost import XGBRegressor
# from lightgbm import LGBMRegressor
# from catboost import CatBoostRegressor
# from sklearn.multioutput import MultiOutputRegressor


# models = {
#     "XGBoost": MultiOutputRegressor(XGBRegressor(
#         n_estimators=500, 
#         learning_rate=0.05,
#         max_depth=6,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         random_state=42,
#         n_jobs=-1
#     )),
#     "LightGBM": MultiOutputRegressor(LGBMRegressor(
#         n_estimators=500,
#         learning_rate=0.05,
#         max_depth=-1,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         random_state=42,
#         n_jobs=-1
#     )),
#     "CatBoost": MultiOutputRegressor(CatBoostRegressor(
#         iterations=500,
#         learning_rate=0.05,
#         depth=6,
#         random_seed=42,
#         verbose=0
#     ))
# }

# # =========================================
# # 3. Train & Evaluate
# # =========================================
# results = []

# for name, model in models.items():
#     print(f"🔄 Training {name} ...")
#     model.fit(X_train, y_train)
#     y_pred = model.predict(X_test)
    
#     mae = mean_absolute_error(y_test, y_pred, multioutput="uniform_average")
#     mse = mean_squared_error(y_test, y_pred, multioutput="uniform_average")
#     r2  = r2_score(y_test, y_pred, multioutput="uniform_average")
    
#     results.append([name, mae, mse, r2])

# # =========================================
# # 4. Show Results
# # =========================================
# results_df = pd.DataFrame(results, columns=["Model", "MAE", "MSE", "R2"])
# print("\n📊 Model Comparison Results")
# print(results_df)

In [ ]:
# # Step 5 — Model Training (Baseline Random Forest)
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.multioutput import MultiOutputRegressor

# # Initialize model
# rf = RandomForestRegressor(
#     n_estimators=200,
#     max_depth=None,
#     random_state=42,
#     n_jobs=-1
# )

# # Wrap in MultiOutputRegressor for multi-target
# model = MultiOutputRegressor(rf)

# # Train the model
# model.fit(X_train, y_train)

# print("✅ Model training completed")


In [ ]:
# # Step 6 — Model Evaluation
# from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
# import numpy as np

# # Predict on test set
# y_pred = model.predict(X_test)

# # Convert ke DataFrame (supaya senang baca)
# y_pred_df = pd.DataFrame(y_pred, columns=target_cols, index=y_test.index)

# # Simpan result untuk setiap minuman
# results = {}

# for col in target_cols:
#     r2 = r2_score(y_test[col], y_pred_df[col])
#     mae = mean_absolute_error(y_test[col], y_pred_df[col])
#     mse = mean_squared_error(y_test[col], y_pred_df[col])
#     results[col] = {"R2": r2, "MAE": mae, "MSE": mse}

# # Tukar ke DataFrame untuk display
# results_df = pd.DataFrame(results).T

# # Kira overall average
# results_df.loc["Average"] = results_df.mean()

# print("✅ Model Evaluation Results")
# results_df
